# Level 1 - Task 1: Data Preprocessing for Machine Learning
**Codveda ML Internship**

**Goal:** turn a raw dataset into something a machine learning model can use.

**Dataset:** Daily stock prices (`symbol, date, open, high, low, close, volume`). It has real missing values and a text column, so it exercises every step of the task.

**Steps:** 1) explore, 2) handle missing data, 3) encode categories, 4) scale numbers, 5) split into train/test.

**Example prediction target:** `close` (the closing price), so the other columns are the features.

In [9]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
DATA_PATH = "../data/stock_prices .csv"   # change this if your file lives elsewhere
df = pd.read_csv(DATA_PATH)
print("Shape (rows, columns):", df.shape)
df.head()

Shape (rows, columns): (497472, 7)


,symbol,date,open,high,low,close,volume
0,AAL,2014-01-02,25.0700,25.8200,25.0600,25.3600,8998943
1,AAPL,2014-01-02,79.3828,79.5756,78.8601,79.0185,58791957
2,AAP,2014-01-02,110.3600,111.8800,109.2900,109.7400,542711
3,ABBV,2014-01-02,52.1200,52.3300,51.5200,51.9800,4569061
4,ABC,2014-01-02,70.1100,70.2300,69.4800,69.8900,1148391


## Step 1: Explore the raw data

In [10]:
df.info()
print()
print("Duplicate rows:", df.duplicated().sum())
df.describe().round(2)

<class 'pandas.DataFrame'>
RangeIndex: 497472 entries, 0 to 497471
Data columns (total 7 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   symbol  497472 non-null  str    
 1   date    497472 non-null  str    
 2   open    497461 non-null  float64
 3   high    497464 non-null  float64
 4   low     497464 non-null  float64
 5   close   497472 non-null  float64
 6   volume  497472 non-null  int64  
dtypes: float64(4), int64(1), str(2)
memory usage: 26.6 MB

Duplicate rows: 0


,open,high,low,close,volume
count,497461.00,497464.00,497464.00,497472.00,4.974720e+05
mean,86.35,87.13,85.55,86.37,4.253611e+06
std,101.47,102.31,100.57,101.47,8.232139e+06
min,1.62,1.69,1.50,1.59,0.000000e+00
25%,41.69,42.09,41.28,41.70,1.080166e+06
50%,64.97,65.56,64.35,64.98,2.084896e+06
75%,98.41,99.23,97.58,98.42,4.271928e+06
max,2044.00,2067.99,2035.11,2049.00,6.182376e+08


**What this means:** `info()` shows each column's type and how many values are filled in. Any column with fewer non-null values than the total has missing data. `describe()` shows that the columns are on very different scales (prices are small, volume is huge), which is why we scale later.

## Step 2: Find the missing data

In [11]:
missing = df.isna().sum()
missing = missing[missing > 0].to_frame("missing_count")
missing["percent"] = (missing["missing_count"] / len(df) * 100).round(4)
print(missing)
print()
print("Total missing cells:", int(df.isna().sum().sum()))
df[df.isna().any(axis=1)].head(8)

      missing_count  percent
open             11   0.0022
high              8   0.0016
low               8   0.0016

Total missing cells: 27


,symbol,date,open,high,low,close,volume
166348,VRTX,2015-05-12,NaN,NaN,NaN,124.08,569747
175557,REGN,2015-06-09,NaN,NaN,NaN,526.09,12135
182011,WRK,2015-06-26,NaN,NaN,NaN,61.90,100
188547,DHR,2015-07-17,NaN,88.76,88.24,88.72,2056819
188578,ES,2015-07-17,NaN,48.49,47.85,47.92,1246786
188760,O,2015-07-17,NaN,47.31,46.83,46.99,1229513
249223,DHR,2016-01-12,NaN,NaN,NaN,88.55,0
249438,O,2016-01-12,NaN,NaN,NaN,52.43,0


**What this means:** Only a tiny fraction of rows have gaps. We have two options:
- **Drop** the rows: fine when very few are missing, but throws data away.
- **Fill** them (impute): keep every row. For numbers, the **median** is safer than the mean because it is not pulled around by extreme values (stock volume has huge spikes).

We fill with the median for numbers and the most frequent value for text columns. The fill values are learned from the **training set only** (see Step 5) to avoid data leakage.

## Step 3: Prepare the columns

In [12]:
# 'date' is text, so turn it into numbers a model can use, then drop the original.
df["date"] = pd.to_datetime(df["date"])
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day_of_week"] = df["date"].dt.dayofweek
df = df.drop(columns=["date"])

y = df["close"]                 # target
X = df.drop(columns=["close"])  # features

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()
print("Numeric columns:    ", num_cols)
print("Categorical columns:", cat_cols)
print("Unique stock symbols:", X["symbol"].nunique())

Numeric columns:     ['open', 'high', 'low', 'volume', 'year', 'month', 'day_of_week']
Categorical columns: ['symbol']
Unique stock symbols: 505


## Step 4: Encoding categorical variables (why one-hot, not label encoding)

In [13]:
# Label encoding turns each symbol into a number: A=0, AAL=1, AAP=2, ...
codes = LabelEncoder().fit_transform(X["symbol"].dropna().head(2000))
print("Label-encoded example:", dict(zip(X["symbol"].dropna().head(2000).unique()[:5], np.unique(codes)[:5])))

Label-encoded example: {'AAL': np.int64(0), 'AAPL': np.int64(1), 'AAP': np.int64(2), 'ABBV': np.int64(3), 'ABC': np.int64(4)}


**What this means:** Label encoding gives each stock a number, but a model would wrongly read that as an order (symbol 2 is "bigger" than symbol 1). Stock symbols have no natural order, so we use **one-hot encoding**: one 0/1 column per symbol. That gives about 500 extra columns, so we keep them in a **sparse matrix**, which only stores the 1s and saves a lot of memory. Label encoding is fine for categories that really are ordered (like Low < Medium < High).

## Step 5: Split first, then impute, encode and scale (no data leakage)

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE)
print("Train rows:", X_train.shape[0], "| Test rows:", X_test.shape[0])

numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale",  StandardScaler()),
])
categorical_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols),
])

# fit on TRAIN only, then apply the same learned rules to TEST
X_train_p = preprocessor.fit_transform(X_train)
X_test_p  = preprocessor.transform(X_test)
print("Processed train shape:", X_train_p.shape)
print("Processed test shape: ", X_test_p.shape)

Train rows: 397977 | Test rows: 99495
Processed train shape: (397977, 512)
Processed test shape:  (99495, 512)


**What this means:** We split *before* learning any fill values or scaling rules. If we had scaled the whole dataset first, information from the test rows would leak into training and make our results look better than they really are. The `Pipeline` + `ColumnTransformer` keeps this correct and repeatable.

## Step 6: Check the result

In [15]:
n = len(num_cols)
scaled_train = X_train_p[:, :n].toarray() if hasattr(X_train_p, "toarray") else X_train_p[:, :n]
check = pd.DataFrame({"mean": scaled_train.mean(axis=0), "std": scaled_train.std(axis=0)}, index=num_cols).round(3)
print("Scaled numeric columns (train): mean should be ~0 and std ~1")
print(check)
print()
before = X_train[num_cols].isna().sum().sum()
after = int(np.isnan(scaled_train).sum())
print("Missing numeric cells in train - before:", int(before), "| after:", after)

Scaled numeric columns (train): mean should be ~0 and std ~1
             mean  std
open          0.0  1.0
high          0.0  1.0
low           0.0  1.0
volume       -0.0  1.0
year         -0.0  1.0
month         0.0  1.0
day_of_week   0.0  1.0

Missing numeric cells in train - before: 24 | after: 0


## Conclusion
- Found and filled the missing values using the median (numbers) learned from the training data.
- Converted `date` into year, month and day-of-week, and one-hot encoded the stock `symbol`.
- Standardised the numeric columns to mean 0 and standard deviation 1.
- Split the data 80/20 into train and test sets with `random_state=42`.
- The data is now ready for a model.

**Note:** This data is a time series. A random split is fine for learning preprocessing, but for real stock forecasting you would split by time (train on earlier dates, test on later ones).